# P1-E16.7D FINAL — Full-Development Selected-Model Refit, Serialization, and Retrospective Locked-Test Evaluation

## Purpose

This experiment consumes the frozen endpoint-specific decisions from P1-E16.7C, refits the selected base models on the complete CAUEEG development partition, serializes the fitted preprocessing and model artifacts, and evaluates them on the official CAUEEG test partition without post-test modification.

### Evidence boundary

- The four frozen endpoints and methods cannot be changed here.
- Candidate configuration, training duration, and binary threshold are derived only from P1-E16.7B-R development evidence.
- The official test partition is used only after all deployment decisions are locked.
- External datasets are prohibited.
- Because earlier project experiments inspected CAUEEG test evidence, this is described as a **retrospective locked-test evaluation under the present frozen contract**, not as a historically untouched test.
- Normal-versus-MCI is cross-sectional discrimination, not conversion prediction.

## 1. Configuration and environment


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, random, sys, time
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (balanced_accuracy_score, accuracy_score, f1_score,
                             matthews_corrcoef, roc_auc_score, confusion_matrix,
                             brier_score_loss, log_loss)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED=20261010
MAX_INPUT_EPOCHS=12
SSL_PRETRAIN_EPOCHS=15
BATCH_SIZE=8
LEARNING_RATE=1e-3
WEIGHT_DECAY=1e-4
N_BOOT=5000
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TASKS={'normal_vs_ad':['Normal','AD'],'mci_vs_ad':['MCI','AD'],
       'normal_vs_mci':['Normal','MCI'],'three_class':['Normal','MCI','AD']}
EXPECTED_SELECTION={'normal_vs_ad':'CEEDNet adaptation','mci_vs_ad':'SSL multiresolution attention',
                    'normal_vs_mci':'SSL multiresolution attention','three_class':'SSL multiresolution attention'}

PROJECT_ROOT=Path(r'D:\EEG_AD_Research\dementia\paper one')
OUTPUTS_ROOT=PROJECT_ROOT/'paper1_outputs'
LEGACY_OUTPUTS_ROOT=Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
E07_ROOT=LEGACY_OUTPUTS_ROOT/'p1_e07_1_full_cohort_reference_preprocessing_automated_qa'
E16_7B_ROOT=OUTPUTS_ROOT/'p1_e16_7b_r_unified_four_endpoint_nested_method_comparison'
E16_7C_ROOT=OUTPUTS_ROOT/'p1_e16_7c_endpoint_selection_cross_fitted_reduced_fusion_freeze'
OUTPUT_ROOT=OUTPUTS_ROOT/'p1_e16_7d_full_development_refit_retrospective_locked_test'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'; MODELS=OUTPUT_ROOT/'models'; LOGS=OUTPUT_ROOT/'logs'; CHECKPOINTS=OUTPUT_ROOT/'checkpoints'
for p in (TABLES,FIGURES,MODELS,LOGS,CHECKPOINTS): p.mkdir(parents=True,exist_ok=True)

EXACT_MANIFEST=E07_ROOT/'tables'/'09_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv'
EXACT_SELECTION=E16_7C_ROOT/'tables'/'06_ENDPOINT_SELECTION_DECISIONS.csv'
EXACT_CONTRACTS=E16_7C_ROOT/'tables'/'07_FROZEN_DEPLOYMENT_SPECIFICATIONS.csv'
EXACT_E16_7C_INTEGRITY=E16_7C_ROOT/'tables'/'08_FINAL_INTEGRITY_CHECKS.csv'
EXACT_E16_7C_RUN=E16_7C_ROOT/'run_record.json'
EXACT_INNER_AUDIT=E16_7B_ROOT/'tables'/'05_INNER_SELECTION_AUDIT.csv'
EXACT_OOF=E16_7B_ROOT/'tables'/'08_ALL_METHOD_NESTED_OOF_PREDICTIONS.csv'

def seed_all(seed):
    seed=int(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
def sha256_file(path,chunk=1<<20):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
def save_table(frame,name):
    path=TABLES/f'{name}.csv'; frame.to_csv(path,index=False); return path
def json_default(x):
    if isinstance(x,(np.integer,)): return int(x)
    if isinstance(x,(np.floating,)): return float(x)
    if isinstance(x,np.ndarray): return x.tolist()
    if isinstance(x,Path): return str(x)
    raise TypeError(type(x).__name__)
seed_all(SEED)
print({'device':str(DEVICE),'torch':torch.__version__,'python':sys.version.split()[0]})

## 2. Authenticate the frozen selection and construct the four endpoint cohorts


In [ ]:
required=[EXACT_MANIFEST,EXACT_SELECTION,EXACT_CONTRACTS,EXACT_E16_7C_INTEGRITY,EXACT_E16_7C_RUN,EXACT_INNER_AUDIT,EXACT_OOF]
missing=[str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('Missing frozen upstream artifact(s):\n'+'\n'.join(missing))

integrity7c=pd.read_csv(EXACT_E16_7C_INTEGRITY)
if not integrity7c['pass'].astype(str).str.lower().eq('true').all(): raise AssertionError('P1-E16.7C integrity did not pass.')
run7c=json.loads(EXACT_E16_7C_RUN.read_text(encoding='utf-8'))
if run7c.get('status')!='PASSED' or not run7c.get('integrity_passed'): raise AssertionError('P1-E16.7C run is not locked.')
if run7c.get('official_test_accessed') or run7c.get('external_data_accessed'): raise AssertionError('P1-E16.7C selection evidence crossed its boundary.')

selection=pd.read_csv(EXACT_SELECTION)
observed=dict(zip(selection.task,selection.selected_method))
if observed!=EXPECTED_SELECTION: raise AssertionError(f'Frozen selection mismatch: {observed}')
contracts=pd.read_csv(EXACT_CONTRACTS)
for r in contracts.itertuples():
    p=Path(r.contract_path)
    if not p.is_file() or sha256_file(p)!=r.contract_sha256: raise AssertionError(f'Contract authentication failed: {p}')

manifest=pd.read_csv(EXACT_MANIFEST,dtype={'recording_id':str})
manifest['recording_id']=manifest.recording_id.astype(str).str.strip().str.zfill(5)
needed={'recording_id','official_split','experimental_label','array_path'}
if not needed.issubset(manifest): raise KeyError(f'Manifest lacks {needed-set(manifest)}')
manifest=manifest.loc[manifest.experimental_label.isin(['Normal','MCI','AD'])].copy()
if 'identity_group' in manifest:
    raw=manifest.identity_group; valid=raw.notna()&raw.astype(str).str.strip().ne('')
    manifest['group_id']=np.where(valid,raw.astype(str),manifest.recording_id)
else: manifest['group_id']=manifest.recording_id
development=manifest.loc[manifest.official_split.astype(str).str.lower().isin(['train','training','validation','val'])].copy()
official_test=manifest.loc[manifest.official_split.astype(str).str.lower().eq('test')].copy()
if set(development.group_id)&set(official_test.group_id): raise AssertionError('Development/test group overlap detected.')

scope=[]
for task,classes in TASKS.items():
    for split,frame in [('development',development),('official_test',official_test)]:
        q=frame.loc[frame.experimental_label.isin(classes)]
        row={'task':task,'split':split,'n':len(q)}; row.update({f'n_{c}':int(q.experimental_label.eq(c).sum()) for c in classes}); scope.append(row)
scope=pd.DataFrame(scope)
if scope.filter(regex='^n_').fillna(1).eq(0).any(axis=1).any(): raise AssertionError('An endpoint split lacks a required class.')
inputs=pd.DataFrame([{'role':p.name,'path':str(p),'sha256':sha256_file(p)} for p in required])
save_table(inputs,'01_AUTHENTICATED_INPUTS'); save_table(scope,'02_ENDPOINT_COHORT_SCOPE')
display(scope); display(selection)

## 3. Freeze final candidate, training duration, and binary decision threshold from development evidence


In [ ]:
inner=pd.read_csv(EXACT_INNER_AUDIT)
oof=pd.read_csv(EXACT_OOF,dtype={'recording_id':str}).rename(columns={'endpoint':'task','fold':'outer_fold'})
oof['recording_id']=oof.recording_id.astype(str).str.strip().str.zfill(5)

def choose_final_recipe(task,method,classes):
    q=inner.loc[(inner.task.eq(task))&(inner.method.eq(method))].copy()
    if q.empty: raise AssertionError(f'No inner-selection evidence for {task}, {method}')
    q=q.sort_values(['outer_fold','inner_balanced_accuracy','inner_roc_auc','candidate'],ascending=[True,False,False,True])
    winners=q.groupby('outer_fold',as_index=False).first()
    candidate_summary=(winners.groupby('candidate').agg(selection_count=('outer_fold','size'),
                       mean_inner_ba=('inner_balanced_accuracy','mean'),mean_inner_auc=('inner_roc_auc','mean'))
                       .reset_index().sort_values(['selection_count','mean_inner_ba','mean_inner_auc','candidate'],ascending=[False,False,False,True]))
    candidate=candidate_summary.iloc[0].candidate
    selected_rows=winners.loc[winners.candidate.eq(candidate)]
    epochs=int(max(1,round(selected_rows.refit_epochs.median())))
    threshold=None
    if len(classes)==2:
        qp=oof.loc[(oof.task.eq(task))&(oof.method.eq(method))].copy()
        y=qp.true_label.map({classes[0]:0,classes[1]:1}).to_numpy(int)
        p=qp[f'probability_{classes[1]}'].to_numpy(float)
        candidates=np.unique(np.r_[.5,p]); scores=[balanced_accuracy_score(y,p>=t) for t in candidates]
        best=max(scores); threshold=float(sorted([t for t,s in zip(candidates,scores) if np.isclose(s,best)],key=lambda t:(abs(t-.5),t))[0])
    return candidate,epochs,threshold,int(len(winners)),candidate_summary

recipes=[]
for task,classes in TASKS.items():
    method=EXPECTED_SELECTION[task]; candidate,epochs,threshold,nfolds,csummary=choose_final_recipe(task,method,classes)
    recipes.append({'task':task,'method':method,'candidate':candidate,'training_epochs':epochs,
                    'binary_threshold':threshold,'outer_selection_folds':nfolds,
                    'candidate_rule':'outer-fold selection frequency; mean inner BA; mean inner AUC; name'})
recipes=pd.DataFrame(recipes)
save_table(recipes,'03_FROZEN_FULL_DEVELOPMENT_RECIPES'); display(recipes)

## 4. Recording loader, training-only normalization, and selected architectures


In [ ]:
def load_epochs(path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(path)
    with np.load(path,allow_pickle=False) as z:
        key=next((k for k in ('epochs_uv','epochs','data','X') if k in z),None)
        if key is None: raise KeyError(f'No epoch array in {path}')
        x=np.asarray(z[key],np.float32)
    if x.ndim!=3: raise ValueError(f'Expected [epochs,channels,time], got {x.shape}')
    if x.shape[1]!=19 and x.shape[2]==19: x=x.transpose(0,2,1)
    if x.shape[1]!=19: raise ValueError(f'Expected 19 channels, got {x.shape}')
    if len(x)>MAX_INPUT_EPOCHS:
        take=np.linspace(0,len(x)-1,MAX_INPUT_EPOCHS).round().astype(int); x=x[take]
    return x
def signal_stats(frame):
    total=np.zeros(19,np.float64); square=np.zeros(19,np.float64); n=0
    for path in frame.array_path:
        x=load_epochs(path); total+=x.sum((0,2),dtype=np.float64); square+=np.square(x,dtype=np.float64).sum((0,2)); n+=x.shape[0]*x.shape[2]
    mean=total/n; sd=np.sqrt(np.maximum(square/n-mean**2,1e-8)); return mean.astype(np.float32),sd.astype(np.float32)
def augment(x):
    if np.random.rand()<.5: x*=np.random.uniform(.90,1.10)
    if np.random.rand()<.5: x+=np.random.normal(0,.03,x.shape).astype(np.float32)
    if np.random.rand()<.3:
        width=max(1,x.shape[-1]//20); start=np.random.randint(0,max(1,x.shape[-1]-width)); x[...,start:start+width]=0
    return x
class RecordingDataset(Dataset):
    def __init__(self,frame,classes,mean,sd,training=False,pair=False):
        self.f=frame.reset_index(drop=True); self.c2i={c:i for i,c in enumerate(classes)}
        self.mean=mean[None,:,None]; self.sd=sd[None,:,None]; self.training=training; self.pair=pair
    def __len__(self): return len(self.f)
    def __getitem__(self,i):
        r=self.f.iloc[i]; x=np.clip((load_epochs(r.array_path)-self.mean)/self.sd,-8,8).astype(np.float32)
        if self.pair:
            j=np.random.randint(len(x)); return torch.from_numpy(augment(x[j].copy())),torch.from_numpy(augment(x[j].copy()))
        if self.training: x=np.stack([augment(e.copy()) for e in x])
        return torch.from_numpy(x),self.c2i[r.experimental_label],str(r.recording_id)
def collate_recordings(batch):
    xs,ys,ids=zip(*batch); m=max(len(x) for x in xs); t=xs[0].shape[-1]
    out=torch.zeros(len(xs),m,19,t); mask=torch.zeros(len(xs),m,dtype=torch.bool)
    for i,x in enumerate(xs): out[i,:len(x)]=x; mask[i,:len(x)]=True
    return out,mask,torch.tensor(ys,dtype=torch.long),ids
def make_loader(ds,shuffle,seed,pair=False):
    g=torch.Generator().manual_seed(int(seed))
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,generator=g,num_workers=0,
                      pin_memory=torch.cuda.is_available(),collate_fn=None if pair else collate_recordings)

class AttentionPool(nn.Module):
    def __init__(self,d): super().__init__(); self.score=nn.Linear(d,1)
    def forward(self,z,mask):
        a=self.score(z).squeeze(-1).masked_fill(~mask,-1e9); a=torch.softmax(a,1); return (z*a.unsqueeze(-1)).sum(1)
class CEEDNet(nn.Module):
    def __init__(self,n_classes,dropout=.25):
        super().__init__(); self.temporal=nn.Sequential(nn.Conv1d(19,38,31,padding=15,groups=19),nn.Conv1d(38,64,1),nn.BatchNorm1d(64),nn.ELU(),
            nn.AvgPool1d(4),nn.Conv1d(64,96,15,padding=7,groups=32),nn.Conv1d(96,96,1),nn.BatchNorm1d(96),nn.ELU(),nn.AdaptiveAvgPool1d(1))
        self.pool=AttentionPool(96); self.head=nn.Sequential(nn.Dropout(dropout),nn.Linear(96,n_classes))
    def forward(self,x,mask):
        b,e,c,t=x.shape; z=self.temporal(x.reshape(b*e,c,t)).squeeze(-1).reshape(b,e,-1); return self.head(self.pool(z,mask))
class MultiResolutionEncoder(nn.Module):
    def __init__(self,d=96):
        super().__init__(); self.branches=nn.ModuleList([nn.Sequential(nn.Conv1d(19,32,k,padding=k//2),nn.BatchNorm1d(32),nn.ELU(),nn.AdaptiveAvgPool1d(1)) for k in (7,31,101)]); self.proj=nn.Linear(96,d)
    def forward(self,x): return F.normalize(self.proj(torch.cat([b(x).squeeze(-1) for b in self.branches],1)),dim=1)
class SSLAttention(nn.Module):
    def __init__(self,n_classes,dropout=.25,encoder_state=None):
        super().__init__(); self.encoder=MultiResolutionEncoder(96)
        if encoder_state is not None: self.encoder.load_state_dict(encoder_state)
        self.pool=AttentionPool(96); self.head=nn.Sequential(nn.Dropout(dropout),nn.Linear(96,n_classes))
    def forward(self,x,mask):
        b,e,c,t=x.shape; z=self.encoder(x.reshape(b*e,c,t)).reshape(b,e,-1); return self.head(self.pool(z,mask))

## 5. Full-development pretraining, supervised refit, prediction, and serialization


In [ ]:
def class_weights(frame,classes):
    counts=frame.experimental_label.value_counts(); return torch.tensor([len(frame)/(len(classes)*counts[c]) for c in classes],dtype=torch.float32,device=DEVICE)
def nt_xent(a,b,temp=.2):
    z=torch.cat([a,b]); sim=z@z.T/temp; n=len(a); sim.fill_diagonal_(-1e9); targets=torch.cat([torch.arange(n,2*n),torch.arange(0,n)]).to(z.device); return F.cross_entropy(sim,targets)
def ssl_pretrain(frame,mean,sd,seed):
    seed_all(seed); enc=MultiResolutionEncoder(96).to(DEVICE); opt=torch.optim.AdamW(enc.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    loader=make_loader(RecordingDataset(frame,['Normal','MCI','AD'],mean,sd,pair=True),True,seed,pair=True)
    for _ in range(SSL_PRETRAIN_EPOCHS):
        enc.train()
        for a,b in loader:
            opt.zero_grad(); loss=nt_xent(enc(a.to(DEVICE)),enc(b.to(DEVICE))); loss.backward(); opt.step()
    return {k:v.detach().cpu().clone() for k,v in enc.state_dict().items()}
def fit_fixed(model,loader,weights,epochs):
    opt=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    history=[]
    for epoch in range(1,int(epochs)+1):
        model.train(); losses=[]
        for x,mask,y,_ in loader:
            opt.zero_grad(); loss=F.cross_entropy(model(x.to(DEVICE),mask.to(DEVICE)),y.to(DEVICE),weight=weights,label_smoothing=.02)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),5); opt.step(); losses.append(float(loss.detach().cpu()))
        history.append({'epoch':epoch,'training_loss':np.mean(losses)})
    return model,pd.DataFrame(history)
def predict(model,loader):
    model.eval(); ids=[]; ys=[]; ps=[]
    with torch.no_grad():
        for x,mask,y,rid in loader:
            ps.append(torch.softmax(model(x.to(DEVICE),mask.to(DEVICE)),1).cpu().numpy()); ys.extend(y.numpy()); ids.extend(rid)
    return np.asarray(ids),np.asarray(ys),np.vstack(ps)
def build_selected(method,candidate,nclasses,ssl_state=None):
    dropout=.50 if candidate=='dropout_0.50' else .25
    if method=='CEEDNet adaptation': return CEEDNet(nclasses,dropout)
    if method=='SSL multiresolution attention': return SSLAttention(nclasses,dropout,ssl_state)
    raise ValueError(f'Unsupported frozen method: {method}')

prediction_rows=[]; artifact_rows=[]; access_rows=[]
for endpoint_index,(task,classes) in enumerate(TASKS.items(),1):
    recipe=recipes.set_index('task').loc[task]; method=recipe.method; candidate=recipe.candidate
    pred_path=CHECKPOINTS/f'{task}__official_test_predictions.csv'; model_path=MODELS/f'{task}__full_development_model.pt'; meta_path=MODELS/f'{task}__full_development_model.json'
    if pred_path.is_file() and model_path.is_file() and meta_path.is_file():
        cached=pd.read_csv(pred_path,dtype={'recording_id':str}); cached['recording_id']=cached.recording_id.astype(str).str.zfill(5)
        expected=set(official_test.loc[official_test.experimental_label.isin(classes),'recording_id'])
        probs=cached[[f'probability_{c}' for c in classes]].to_numpy(float)
        if set(cached.recording_id)==expected and not cached.recording_id.duplicated().any() and np.isfinite(probs).all() and np.allclose(probs.sum(1),1,atol=1e-6):
            prediction_rows.extend(cached.to_dict('records')); meta=json.loads(meta_path.read_text(encoding='utf-8'))
            artifact_rows.append(meta['artifact_record']); access_rows.append(meta['access_record']); print('restored and verified',task); continue
        raise RuntimeError(f'Incomplete frozen checkpoint for {task}; preserve it for audit and investigate before rerunning.')

    train=development.loc[development.experimental_label.isin(classes)].copy(); test=official_test.loc[official_test.experimental_label.isin(classes)].copy()
    mean,sd=signal_stats(train); ssl_state=None
    if method=='SSL multiresolution attention': ssl_state=ssl_pretrain(train,mean,sd,SEED+100000*endpoint_index)
    seed_all(SEED+1000000*endpoint_index); model=build_selected(method,candidate,len(classes),ssl_state).to(DEVICE)
    loader=make_loader(RecordingDataset(train,classes,mean,sd,True),True,SEED+endpoint_index)
    started=time.perf_counter(); model,history=fit_fixed(model,loader,class_weights(train,classes),int(recipe.training_epochs)); runtime=time.perf_counter()-started
    history.to_csv(LOGS/f'{task}__full_development_training_history.csv',index=False)
    state={'state_dict':model.state_dict(),'task':task,'classes':classes,'method':method,'candidate':candidate,
           'training_epochs':int(recipe.training_epochs),'threshold':None if pd.isna(recipe.binary_threshold) else float(recipe.binary_threshold),
           'channel_mean':mean,'channel_sd':sd,'max_input_epochs':MAX_INPUT_EPOCHS,'seed':SEED}
    torch.save(state,model_path)
    model_hash_before_test=sha256_file(model_path)
    access_record={'task':task,'contract_locked_before_test':True,'model_sha256_before_test':model_hash_before_test,
                   'official_test_access_utc':datetime.now(timezone.utc).isoformat(),'access_scope':'retrospective locked-test under P1-E16.7C contract'}
    test_loader=make_loader(RecordingDataset(test,classes,mean,sd,False),False,SEED)
    ids,yt,pt=predict(model,test_loader); threshold=state['threshold']
    yp=(pt[:,1]>=threshold).astype(int) if len(classes)==2 else pt.argmax(1)
    local=[]
    for i,rid in enumerate(ids):
        row={'task':task,'recording_id':rid,'true_label':classes[yt[i]],'predicted_label':classes[yp[i]],
             'method':method,'candidate':candidate,'threshold':threshold}
        row.update({f'probability_{c}':float(pt[i,j]) for j,c in enumerate(classes)}); local.append(row)
    tmp=pred_path.with_suffix('.tmp'); pd.DataFrame(local).to_csv(tmp,index=False); os.replace(tmp,pred_path)
    if sha256_file(model_path)!=model_hash_before_test: raise AssertionError('Model changed after official-test prediction.')
    artifact={'task':task,'method':method,'candidate':candidate,'model_path':str(model_path),'model_sha256':model_hash_before_test,
              'n_parameters':int(sum(p.numel() for p in model.parameters())),'training_seconds':runtime,
              'training_epochs':int(recipe.training_epochs),'binary_threshold':threshold}
    meta={'artifact_record':artifact,'access_record':access_record}
    meta_path.write_text(json.dumps(meta,indent=2,default=json_default),encoding='utf-8')
    prediction_rows.extend(local); artifact_rows.append(artifact); access_rows.append(access_record); print('complete',task)

predictions=pd.DataFrame(prediction_rows); artifacts=pd.DataFrame(artifact_rows); access=pd.DataFrame(access_rows)
save_table(predictions,'04_RETROSPECTIVE_LOCKED_TEST_PREDICTIONS'); save_table(artifacts,'05_SERIALIZED_MODEL_ARTIFACTS'); save_table(access,'06_OFFICIAL_TEST_ACCESS_AUDIT')
display(artifacts); display(access)

## 6. Locked-test metrics, calibration, and recording-level uncertainty


In [ ]:
def ece_score(y,p,n_bins=10):
    confidence=p.max(1); predicted=p.argmax(1); correct=(predicted==y).astype(float); edges=np.linspace(0,1,n_bins+1); value=0.0
    for lo,hi in zip(edges[:-1],edges[1:]):
        use=(confidence>=lo)&((confidence<hi) if hi<1 else (confidence<=hi))
        if use.any(): value+=use.mean()*abs(correct[use].mean()-confidence[use].mean())
    return value
def evaluate(q,task):
    classes=TASKS[task]; c2i={c:i for i,c in enumerate(classes)}; y=q.true_label.map(c2i).to_numpy(int); yp=q.predicted_label.map(c2i).to_numpy(int)
    p=q[[f'probability_{c}' for c in classes]].to_numpy(float)
    out={'balanced_accuracy':balanced_accuracy_score(y,yp),'accuracy':accuracy_score(y,yp),'macro_f1':f1_score(y,yp,average='macro'),
         'mcc':matthews_corrcoef(y,yp),'roc_auc':roc_auc_score(y,p[:,1]) if len(classes)==2 else roc_auc_score(y,p,multi_class='ovr',average='macro',labels=np.arange(len(classes))),
         'brier_score':brier_score_loss(y,p[:,1]) if len(classes)==2 else np.mean(np.sum((p-np.eye(len(classes))[y])**2,axis=1)),
         'log_loss':log_loss(y,p,labels=np.arange(len(classes))),'ece_10bin':ece_score(y,p)}
    if len(classes)==2:
        tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel(); out.update({'sensitivity':tp/(tp+fn),'specificity':tn/(tn+fp)})
    return out

rng=np.random.default_rng(SEED); metric_rows=[]; ci_rows=[]
for task,classes in TASKS.items():
    q=predictions.loc[predictions.task.eq(task)].reset_index(drop=True); observed=evaluate(q,task); metric_rows.append({'task':task,'n':len(q),**observed})
    strata={c:np.flatnonzero(q.true_label.to_numpy()==c) for c in classes}; boots=[]
    for _ in range(N_BOOT):
        ix=np.concatenate([rng.choice(strata[c],len(strata[c]),replace=True) for c in classes]); boots.append(evaluate(q.iloc[ix],task))
    for metric in observed:
        values=np.asarray([b[metric] for b in boots],float); values=values[np.isfinite(values)]
        ci_rows.append({'task':task,'metric':metric,'estimate':observed[metric],
                        'ci_low':np.quantile(values,.025) if len(values) else np.nan,'ci_high':np.quantile(values,.975) if len(values) else np.nan,
                        'interval':'class-stratified recording bootstrap percentile','repetitions':N_BOOT})
metrics=pd.DataFrame(metric_rows); intervals=pd.DataFrame(ci_rows)
save_table(metrics,'07_RETROSPECTIVE_LOCKED_TEST_METRICS'); save_table(intervals,'08_RETROSPECTIVE_LOCKED_TEST_INTERVALS')
display(metrics.round(4)); display(intervals.loc[intervals.metric.isin(['balanced_accuracy','roc_auc','sensitivity','specificity'])].round(4))

## 7. Confusion matrices and publication-ready figures


In [ ]:
labels={'normal_vs_ad':'Normal vs AD','mci_vs_ad':'MCI vs AD','normal_vs_mci':'Normal vs MCI','three_class':'Three class'}
fig,axes=plt.subplots(2,2,figsize=(11,9))
for ax,(task,classes) in zip(axes.flat,TASKS.items()):
    q=predictions.loc[predictions.task.eq(task)]; cm=confusion_matrix(q.true_label,q.predicted_label,labels=classes)
    image=ax.imshow(cm,cmap='Blues')
    for i in range(len(classes)):
        for j in range(len(classes)): ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes); ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Observed'); ax.set_title(labels[task])
fig.suptitle('Retrospective locked-test confusion matrices',fontweight='bold'); fig.tight_layout()
fig.savefig(FIGURES/'Figure_P1_E16_7D_Locked_Test_Confusion_Matrices.pdf',bbox_inches='tight'); fig.savefig(FIGURES/'Figure_P1_E16_7D_Locked_Test_Confusion_Matrices.png',dpi=400,bbox_inches='tight'); plt.show()

q=intervals.loc[intervals.metric.eq('balanced_accuracy')].set_index('task').reindex(TASKS)
x=np.arange(len(q)); y=q.estimate.to_numpy(); lo=y-q.ci_low.to_numpy(); hi=q.ci_high.to_numpy()-y
fig,ax=plt.subplots(figsize=(9,5)); ax.errorbar(x,y,yerr=[lo,hi],fmt='o',capsize=5,color='#1f4e79')
for i,(task,classes) in enumerate(TASKS.items()): ax.hlines(1/len(classes),i-.32,i+.32,colors='#b22222',linestyles='--')
ax.set_xticks(x); ax.set_xticklabels([labels[t] for t in TASKS]); ax.set_ylabel('Balanced accuracy (95% bootstrap CI)'); ax.set_ylim(.2,1.02); ax.grid(axis='y',alpha=.25)
ax.set_title('Frozen selected models on the retrospective CAUEEG locked test',fontweight='bold'); fig.tight_layout()
fig.savefig(FIGURES/'Figure_P1_E16_7D_Locked_Test_Balanced_Accuracy.pdf',bbox_inches='tight'); fig.savefig(FIGURES/'Figure_P1_E16_7D_Locked_Test_Balanced_Accuracy.png',dpi=400,bbox_inches='tight'); plt.show()

## 8. Final integrity and provenance lock


In [ ]:
checks=[]
for task,classes in TASKS.items():
    q=predictions.loc[predictions.task.eq(task)]; expected=official_test.loc[official_test.experimental_label.isin(classes)]
    p=q[[f'probability_{c}' for c in classes]].to_numpy(float)
    artifact=artifacts.loc[artifacts.task.eq(task)].iloc[0]; recipe=recipes.loc[recipes.task.eq(task)].iloc[0]
    checks.extend([
        {'scope':task,'check':'one prediction per eligible official-test recording','pass':len(q)==len(expected) and not q.recording_id.duplicated().any() and set(q.recording_id)==set(expected.recording_id)},
        {'scope':task,'check':'probabilities finite and sum to one','pass':np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-6)},
        {'scope':task,'check':'selected method matches frozen contract','pass':q.method.eq(EXPECTED_SELECTION[task]).all()},
        {'scope':task,'check':'serialized model hash unchanged','pass':Path(artifact.model_path).is_file() and sha256_file(Path(artifact.model_path))==artifact.model_sha256},
        {'scope':task,'check':'frozen candidate and threshold applied','pass':q.candidate.eq(recipe.candidate).all() and (len(classes)==3 or np.allclose(q.threshold.astype(float),float(recipe.binary_threshold)))},
        {'scope':task,'check':'correct mathematical chance reference','pass':np.isclose(1/len(classes),.5 if len(classes)==2 else 1/3)},])
checks.extend([
    {'scope':'all','check':'four endpoints complete','pass':set(predictions.task)==set(TASKS)},
    {'scope':'all','check':'P1-E16.7C contracts authenticated','pass':True},
    {'scope':'all','check':'model artifacts frozen before test prediction','pass':access.contract_locked_before_test.all()},
    {'scope':'all','check':'external datasets excluded','pass':True},
    {'scope':'all','check':'no post-test tuning or recalibration','pass':True},
    {'scope':'all','check':'test evidence labelled retrospective locked test','pass':access.access_scope.str.contains('retrospective locked-test',regex=False).all()},
    {'scope':'all','check':'no target performance used','pass':True},])
checks=pd.DataFrame(checks); save_table(checks,'09_FINAL_INTEGRITY_CHECKS'); display(checks)
if not checks['pass'].all(): raise AssertionError('P1-E16.7D integrity failure.')

run={'experiment':'P1-E16.7D full-development selected-model refit and retrospective locked-test evaluation','status':'PASSED',
     'completed_utc':datetime.now(timezone.utc).isoformat(),'evidence_label':'retrospective locked test under P1-E16.7C frozen contract',
     'historically_untouched_claim':False,'endpoints':TASKS,'selected_methods':EXPECTED_SELECTION,'recipes':recipes.to_dict('records'),
     'metrics':metrics.to_dict('records'),'model_artifacts':artifacts.to_dict('records'),'external_data_accessed':False,
     'post_test_tuning':False,'performance_target_used':False,'integrity_passed':True,'input_files':inputs.to_dict('records'),
     'environment':{'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'device':str(DEVICE)}}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run,indent=2,default=json_default),encoding='utf-8')
print('P1-E16.7D PASSED — FOUR SELECTED MODELS REFITTED, SERIALIZED, AND RETROSPECTIVELY LOCK-TESTED')

## 9. Interpretation guardrails and next experiment

1. Report these values as retrospective official CAUEEG locked-test evidence under the P1-E16.7C contract.
2. Do not call the CAUEEG test historically untouched or prospectively prespecified.
3. Do not modify models, thresholds, preprocessing, or endpoint definitions after observing these results.
4. Small endpoint-specific test samples require confidence intervals and restrained interpretation.
5. The next experiment is **P1-E16.7E — Frozen External Validation and Harmonization Audit**: AHEPA supports Normal versus AD; Dryad/P-ADIC supports the compatible binary contrasts and three-class endpoint. Earlier external inspections must be disclosed chronologically, and the evaluation must not be described as untouched.